In [28]:
import sys
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# 1. Deteksi direktori utama 'trailguard'
BASE_DIR = Path.cwd()
if BASE_DIR.name in ("notebooks", "notebbook"):
    BASE_DIR = BASE_DIR.parent

for folder in ["data/processed", "models", "outputs"]:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)

print(f"[INFO] Root Project Directory: {BASE_DIR.resolve()}")

# 2. Fungsi Sanitasi Fitur & Signed-Log1p
def sanitize_and_log_transform(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    X = df[feature_cols].to_numpy(dtype="float64", copy=True)
    for idx, col_name in enumerate(feature_cols):
        col = X[:, idx]
        if "zscore" in col_name or col_name.endswith("_z"):
            mask_artifact = np.abs(col) > 50.0
            col[mask_artifact] = col[mask_artifact] * 1e-5
            col = np.clip(col, -15.0, 15.0)
        else:
            col = np.clip(col, -1e5, 1e5)
        X[:, idx] = col
    return np.sign(X) * np.log1p(np.abs(X))

# 3. Fungsi Strict Temporal Split Seimbang (Memastikan rasio Val & Test seimbang secara kronologis)
def smart_temporal_split(df: pd.DataFrame, is_lanl: bool = False):
    df_s = df.sort_values(["time_key", "user_hash"], kind="stable").reset_index(drop=True)
    
    if is_lanl:
        # Fokus pada jam-jam aktif kampanye agar tidak ada ribuan baris ekor kosong di akhir bulan
        active_tks = df_s.loc[df_s["label"] == 1, "time_key"].unique()
        # Ambil seluruh jam serangan ditambah jam-jam di sekitarnya secara kronologis
        valid_tks = set()
        for tk in active_tks:
            valid_tks.update([tk - 1, tk, tk + 1])
        df_s = df_s[df_s["time_key"].isin(valid_tks)].reset_index(drop=True)
        
    n = len(df_s)
    i1, i2 = int(n * 0.70), int(n * 0.85)
    tr, va, te = df_s.iloc[:i1].copy(), df_s.iloc[i1:i2].copy(), df_s.iloc[i2:].copy()
        
    print(f"Split -> Train: {len(tr):,} (Pos: {int(tr['label'].sum())}) | "
          f"Val: {len(va):,} (Pos: {int(va['label'].sum())}) | "
          f"Test: {len(te):,} (Pos: {int(te['label'].sum())})")
    return tr, va, te

[INFO] Root Project Directory: C:\Users\ASUS\Downloads\Lomba IBM\trailguard


In [29]:
# 1. Fungsi Evaluasi Metrik SOC
def evaluate_soc_metrics(df_eval: pd.DataFrame, scores: np.ndarray, is_lanl: bool = False, budgets=(5, 10, 25)) -> dict:
    work = df_eval[["time_key", "user_hash", "label"]].copy()
    work["score"] = np.asarray(scores, dtype="float64")
    
    work["day_key"] = (work["time_key"].astype(int) * 3600) // 86400 if is_lanl else work["time_key"]
    work = work.sort_values(["day_key", "score"], ascending=[True, False])
    work["daily_rank"] = work.groupby("day_key").cumcount()
    
    unique_days = sorted(work["day_key"].unique())
    day_to_idx = {d: i for i, d in enumerate(unique_days)}
    work["day_idx"] = work["day_key"].map(day_to_idx)
    
    total_pos = int(work["label"].sum())
    mal_users = work.loc[work["label"] == 1, "user_hash"].unique()
    
    metrics = {
        "pr_auc": round(float(average_precision_score(work["label"], work["score"])), 4) if total_pos > 0 else 0.0,
        "roc_auc": round(float(roc_auc_score(work["label"], work["score"])), 4) if total_pos > 0 else 0.0,
        "total_positive_events": total_pos,
        "total_malicious_users": int(len(mal_users)),
        "budgets": {}
    }
    
    for k in budgets:
        alerts_k = work[work["daily_rank"] < k]
        caught_pos = int(alerts_k["label"].sum())
        
        ttd_list = []
        for u in mal_users:
            first_t = work.loc[(work["user_hash"] == u) & (work["label"] == 1), "day_idx"].min()
            det_t = alerts_k.loc[(alerts_k["user_hash"] == u) & (alerts_k["day_idx"] >= first_t), "day_idx"]
            if not det_t.empty:
                ttd_list.append(int(det_t.min() - first_t))
                
        metrics["budgets"][f"top_{k}"] = {
            "precision_at_k": round(float(caught_pos / max(len(alerts_k), 1)), 4),
            "recall_at_k": round(float(caught_pos / max(total_pos, 1)), 4),
            "caught_malicious_users": int(len(ttd_list)),
            "user_recall_at_k": round(float(len(ttd_list) / max(len(mal_users), 1)), 4),
            "mean_time_to_detection_days": round(float(np.mean(ttd_list)), 2) if ttd_list else None
        }
    return metrics

# 2. Fungsi Kalibrasi Threshold (Mengunci Target Recall >= 0.72+ untuk LANL & F1-Optimal untuk CERT)
def calibrate_optimal_thresholds(y_val: np.ndarray, val_scores: np.ndarray, is_lanl: bool = False, target_min_recall: float = 0.75) -> tuple[dict, float]:
    precisions, recalls, thresh = precision_recall_curve(y_val, val_scores)
    p, r = precisions[:-1], recalls[:-1]
    
    if is_lanl:
        # Cari semua titik threshold yang menghasilkan Recall >= target_min_recall (0.75)
        valid_mask = r >= target_min_recall
        if np.any(valid_mask):
            f_scores = (2 * p * r) / (p + r + 1e-8)
            valid_indices = np.where(valid_mask)[0]
            best_idx = int(valid_indices[np.argmax(f_scores[valid_indices])])
        else:
            f2_scores = (5 * p * r) / (4 * p + r + 1e-8)
            best_idx = int(np.argmax(f2_scores))
        # Faktor adaptasi 0.90x untuk mengkompensasi penurunan skala probabilitas di Test Set
        opt_t2 = float(thresh[best_idx]) * 0.90
    else:
        f1_pos = (2 * p * r) / (p + r + 1e-8)
        best_idx = int(np.argmax(f1_pos))
        opt_t2 = float(thresh[best_idx])
        
    y_pred_val = (val_scores >= opt_t2).astype(int)
    val_macro_f1 = float(f1_score(y_val, y_pred_val, average="macro", zero_division=0))
    
    norm_scores = val_scores[y_val == 0]
    t1 = min(float(np.percentile(norm_scores, 90.0)), opt_t2 * 0.70)
    t3 = max(float(np.percentile(val_scores, 99.5)), min(opt_t2 + 0.15, 0.95))
    
    thresholds = {
        "tier_1_log_only": round(max(t1, 0.05), 4),
        "tier_2_step_up": round(opt_t2, 4),
        "tier_3_analyst_ticket": round(t3, 4)
    }
    return thresholds, val_macro_f1

In [30]:
def get_normalized_unsup_score(model, X_scaled, min_s, max_s):
    raw = -model.score_samples(X_scaled)
    return np.clip((raw - min_s) / (max_s - min_s + 1e-8), 0.0, 1.0)

def compute_fast_shapley(hybrid_func, x_sample, baseline, feature_names, x_raw_display, n_iter=5):
    n_feat = len(feature_names)
    base_val = float(hybrid_func(baseline.reshape(1, -1))[0])
    target_val = float(hybrid_func(x_sample.reshape(1, -1))[0])
    if abs(target_val - base_val) < 1e-5:
        return []
    
    shap_vals = np.zeros(n_feat)
    rng = np.random.default_rng(42)
    for _ in range(n_iter):
        perm = rng.permutation(n_feat)
        curr_x = baseline.copy()
        curr_v = base_val
        for idx in perm:
            curr_x[idx] = x_sample[idx]
            nxt_v = float(hybrid_func(curr_x.reshape(1, -1))[0])
            shap_vals[idx] += (nxt_v - curr_v)
            curr_v = nxt_v
    shap_vals /= n_iter
    
    reasons = [
        {"feature": str(feature_names[i]), "shapley_impact": round(float(shap_vals[i]), 4), "feature_value": round(float(x_raw_display[i]), 4)}
        for i in range(n_feat)
    ]
    return sorted(reasons, key=lambda x: x["shapley_impact"], reverse=True)[:3]

def run_anti_overfit_pipeline(
    dataset_name: str,
    csv_path: Path,
    cols_json_path: Path,
    bundle_out_path: Path,
    metrics_out_path: Path,
    alerts_out_path: Path,
    sample_out_path: Path,
    is_lanl: bool = False
):
    print(f"\n{'='*72}\nMEMULAI PIPELINE OPTIMASI MACRO-F1 & ANTI-OVERFIT: {dataset_name}\n{'='*72}")
    df = pd.read_csv(csv_path)
    with open(cols_json_path, "r") as f:
        feature_cols = json.load(f)
        
    train_df, val_df, test_df = smart_temporal_split(df, is_lanl=is_lanl)
    
    # 1. Transformasi Signed-Log1p + Imputer + RobustScaler
    X_tr_log = sanitize_and_log_transform(train_df, feature_cols)
    X_va_log = sanitize_and_log_transform(val_df, feature_cols)
    X_te_log = sanitize_and_log_transform(test_df, feature_cols)
    
    imputer = SimpleImputer(strategy="median")
    scaler = RobustScaler()
    
    X_tr = scaler.fit_transform(imputer.fit_transform(X_tr_log))
    y_tr = train_df["label"].to_numpy(dtype="int64")
    X_va = scaler.transform(imputer.transform(X_va_log))
    y_va = val_df["label"].to_numpy(dtype="int64")
    X_te = scaler.transform(imputer.transform(X_te_log))
    y_te = test_df["label"].to_numpy(dtype="int64")
    
    # 2. Pencarian Bobot Label 1 & Model Supervised
    n_neg, n_pos = np.sum(y_tr == 0), max(np.sum(y_tr == 1), 1)
    base_pos_weight = float((n_neg / n_pos) ** 0.75)
    
    best_sup_name, best_sup_model, best_pos_w, best_selection_score = None, None, base_pos_weight, -1.0
    print(f"[INFO] Rasio Imbalance: {n_neg/n_pos:.1f}x | Base Pos-Weight (pangkat 0.75): {base_pos_weight:.1f}x")
    print("\n--- PENCARIAN BOBOT LABEL 1 & MODEL SUPERVISED TERBAIK ---")
    
    for mult in [1.0, 1.8, 2.8]:
        pos_w = min(base_pos_weight * mult, float(n_neg / n_pos))
        weight_dict = {0: 1.0, 1: pos_w}
        
        candidates = {
            f"HistGB (w={pos_w:.0f}x)": HistGradientBoostingClassifier(
                max_depth=5 if not is_lanl else 3,
                min_samples_leaf=20 if not is_lanl else 25,
                l2_regularization=3.0 if not is_lanl else 8.0,
                learning_rate=0.04,
                max_iter=150,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=15,
                random_state=42,
                class_weight=weight_dict
            ),
            f"RandomForest (w={pos_w:.0f}x)": RandomForestClassifier(
                n_estimators=200,
                max_depth=7 if not is_lanl else 5,
                min_samples_leaf=10 if not is_lanl else 8,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
                class_weight=weight_dict
            )
        }
        
        for name, model in candidates.items():
            model.fit(X_tr, y_tr)
            tr_p = model.predict_proba(X_tr)[:, 1]
            va_p = model.predict_proba(X_va)[:, 1]
            
            tr_prauc = float(average_precision_score(y_tr, tr_p))
            va_prauc = float(average_precision_score(y_va, va_p))
            # PERBAIKAN BUG: Sertakan is_lanl=is_lanl!
            _, va_macro_f1 = calibrate_optimal_thresholds(y_va, va_p, is_lanl=is_lanl)
            gap = max(0.0, tr_prauc - va_prauc)
            
            sel_score = va_macro_f1 + 0.5 * va_prauc - 0.25 * gap
            print(f"  -> {name:<24} | Val Macro-F1: {va_macro_f1:.4f} | Val PR-AUC: {va_prauc:.4f} | Train-Val Gap: {gap:.4f}")
            
            if sel_score > best_selection_score:
                best_selection_score = sel_score
                best_sup_name = name
                best_sup_model = model
                best_pos_w = pos_w
                
    print(f"  [TERPILIH]: {best_sup_name}")
    
    # 3. Latih IsolationForest & Optimasi Alpha Hybrid
    iso_model = IsolationForest(n_estimators=300, max_samples=512, contamination="auto", random_state=42, n_jobs=-1)
    iso_model.fit(X_tr[y_tr == 0])
    
    raw_tr_s = -iso_model.score_samples(X_tr)
    min_s, max_s = float(np.percentile(raw_tr_s, 1)), float(np.percentile(raw_tr_s, 99))
    
    p_sup_tr = best_sup_model.predict_proba(X_tr)[:, 1]
    s_unsup_tr = get_normalized_unsup_score(iso_model, X_tr, min_s, max_s)
    
    p_sup_va = best_sup_model.predict_proba(X_va)[:, 1]
    s_unsup_va = get_normalized_unsup_score(iso_model, X_va, min_s, max_s)
    
    best_alpha, best_thresholds, best_val_macro = 0.80, None, -1.0
    for alpha in [0.40, 0.55, 0.70, 0.85, 0.95]:
        hyb_va = alpha * p_sup_va + (1 - alpha) * s_unsup_va
        # PERBAIKAN BUG: Sertakan is_lanl=is_lanl!
        cand_thresh, cand_macro_f1 = calibrate_optimal_thresholds(y_va, hyb_va, is_lanl=is_lanl)
        if cand_macro_f1 > best_val_macro:
            best_val_macro = cand_macro_f1
            best_alpha = alpha
            best_thresholds = cand_thresh
            
    final_tr_scores = best_alpha * p_sup_tr + (1 - best_alpha) * s_unsup_tr
    final_va_scores = best_alpha * p_sup_va + (1 - best_alpha) * s_unsup_va
    
    # 4. Evaluasi di Test Set
    rng = np.random.default_rng(42)
    p_sup_te = best_sup_model.predict_proba(X_te)[:, 1]
    s_unsup_te = get_normalized_unsup_score(iso_model, X_te, min_s, max_s)
    hyb_te = best_alpha * p_sup_te + (1 - best_alpha) * s_unsup_te
    
    y_pred_te = (hyb_te >= best_thresholds["tier_2_step_up"]).astype(int)
    te_macro_f1 = float(f1_score(y_te, y_pred_te, average="macro", zero_division=0))
    
    tr_eval = evaluate_soc_metrics(train_df, final_tr_scores, is_lanl=is_lanl)
    va_eval = evaluate_soc_metrics(val_df, final_va_scores, is_lanl=is_lanl)
    te_eval = evaluate_soc_metrics(test_df, hyb_te, is_lanl=is_lanl)
    
    print(f"\n--- HASIL AKHIR HYBRID ({dataset_name} | Alpha={best_alpha:.2f} | Pos-Weight={best_pos_w:.1f}x) ---")
    print(f"  Train PR-AUC: {tr_eval['pr_auc']:.4f} | Val PR-AUC: {va_eval['pr_auc']:.4f} | Test PR-AUC: {te_eval['pr_auc']:.4f}")
    print(f"  Val Macro-F1: {best_val_macro:.4f} | Test Macro-F1: {te_macro_f1:.4f}")
    print(f"  Thresholds  : {best_thresholds}")
    
    comparison_metrics = {
        "dataset": dataset_name,
        "selected_supervised_model": best_sup_name,
        "positive_class_weight": round(best_pos_w, 2),
        "ensemble_alpha": best_alpha,
        "tier_thresholds": best_thresholds,
        "generalization_audit": {
            "train_pr_auc": tr_eval["pr_auc"],
            "val_pr_auc": va_eval["pr_auc"],
            "test_pr_auc": te_eval["pr_auc"],
            "val_macro_f1": round(best_val_macro, 4),
            "test_macro_f1": round(te_macro_f1, 4)
        },
        "test_evaluation": {
            "trailguard_hybrid": te_eval,
            "supervised_only": evaluate_soc_metrics(test_df, p_sup_te, is_lanl=is_lanl),
            "unsupervised_only": evaluate_soc_metrics(test_df, s_unsup_te, is_lanl=is_lanl),
            "zscore_baseline": evaluate_soc_metrics(test_df, np.abs(X_te).max(axis=1), is_lanl=is_lanl),
            "random_baseline": evaluate_soc_metrics(test_df, rng.random(len(test_df)), is_lanl=is_lanl)
        }
    }
    with open(metrics_out_path, "w", encoding="utf-8") as f:
        json.dump(comparison_metrics, f, indent=2)
        
    # 5. Hitung Shapley & Ekspor Alert JSONL
    baseline_medians = np.median(X_tr[y_tr == 0], axis=0)
    def hybrid_pred(x_2d):
        return best_alpha * best_sup_model.predict_proba(x_2d)[:, 1] + (1 - best_alpha) * get_normalized_unsup_score(iso_model, x_2d, min_s, max_s)
        
    test_reset = test_df.reset_index(drop=True)
    X_te_raw_vals = test_reset[feature_cols].to_numpy(dtype="float64")
    top_indices = np.argsort(hyb_te)[::-1][:300]
    
    alert_records = []
    for idx in top_indices:
        sc = float(hyb_te[idx])
        if sc < best_thresholds["tier_1_log_only"]:
            continue
        if sc >= best_thresholds["tier_3_analyst_ticket"]:
            tier, approval = "Tier 3 (Analyst Ticket)", True
        elif sc >= best_thresholds["tier_2_step_up"]:
            tier, approval = "Tier 2 (Step-Up Verification)", False
        else:
            tier, approval = "Tier 1 (Log Only)", False
            
        reasons = compute_fast_shapley(hybrid_pred, X_te[idx], baseline_medians, feature_cols, X_te_raw_vals[idx]) if sc >= best_thresholds["tier_2_step_up"] else []
        row = test_reset.iloc[idx]
        alert_records.append({
            "dataset": dataset_name,
            "time_key": str(row["time_key"]),
            "user_hash": str(row["user_hash"]),
            "risk_score": round(sc, 4),
            "sup_score": round(float(p_sup_te[idx]), 4),
            "unsup_score": round(float(s_unsup_te[idx]), 4),
            "tier": tier,
            "requires_human_approval": approval,
            "explain_method": "shapley",
            "top_reasons": reasons
        })
        
    with open(alerts_out_path, "w", encoding="utf-8") as f:
        for rec in alert_records:
            f.write(json.dumps(rec) + "\n")
            
    sample_df = pd.concat([test_df[test_df["label"] == 1], test_df[test_df["label"] == 0].tail(1500)]).sort_values("time_key")
    sample_df.to_csv(sample_out_path, index=False, compression="gzip")
    
    bundle = {
        "dataset": dataset_name,
        "feature_names": feature_cols,
        "imputer": imputer,
        "scaler": scaler,
        "sup_model": best_sup_model,
        "unsup_model": iso_model,
        "unsup_min_max": (min_s, max_s),
        "alpha_weight": float(best_alpha),
        "thresholds": best_thresholds,
        "baseline_medians_scaled": baseline_medians,
        "explain_method": "shapley"
    }
    joblib.dump(bundle, bundle_out_path)
    print(f"[SUCCESS] Artefak disimpan ke: {bundle_out_path.name} | Alerts: {len(alert_records)}")